# Assignment 16: SVM, Trees, Ensembles, Validation & Unsupervised Learning
### Author: Parth Dadhaniya

**Dataset:** Heart Disease Dataset (Cleveland / UCI)
**Kaggle Link:** https://www.kaggle.com/datasets/ronitf/heart-disease-uci

### Step 1: Load and Inspect Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# load dataset
df = pd.read_csv('heart_disease.csv')
print(df.head())
print('Dataset shape:', df.shape)

### Task 1: Support Vector Machine (SVM)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X = df[['age', 'trestbps', 'chol', 'thalach', 'oldpeak']]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# scaling features for SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. linear kernel
svm_linear = SVC(kernel='linear', random_state=42)
svm_linear.fit(X_train_scaled, y_train)

# 2. rbf kernel
svm_rbf = SVC(kernel='rbf', random_state=42)
svm_rbf.fit(X_train_scaled, y_train)

print('Linear Kernel Accuracy:', round(svm_linear.score(X_test_scaled, y_test), 2))
print('RBF Kernel Accuracy:   ', round(svm_rbf.score(X_test_scaled, y_test), 2))

### Task 2: Decision Tree Algorithm

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# small tree for visualization
dt = DecisionTreeClassifier(max_depth=2, random_state=42)
dt.fit(X_train, y_train)

plt.figure(figsize=(8, 5))
plot_tree(dt, feature_names=list(X.columns), class_names=['No Disease', 'Disease'], filled=True)
plt.title('Decision Tree Structure (depth=2)')
plt.show()

# comparing underfitting vs overfitting
underfit_model = DecisionTreeClassifier(max_depth=1, random_state=42).fit(X_train, y_train)
overfit_model = DecisionTreeClassifier(max_depth=None, random_state=42).fit(X_train, y_train)

print('Underfitting (depth=1)   - Train:', round(underfit_model.score(X_train, y_train), 2), '| Test:', round(underfit_model.score(X_test, y_test), 2))
print('Overfitting  (depth=None) - Train:', round(overfit_model.score(X_train, y_train), 2), '| Test:', round(overfit_model.score(X_test, y_test), 2))

### Task 3: Train vs Validation vs Test Split

In [ ]:
# split 60% train, 20% validation, 20% test
X_temp, X_test3, y_temp, y_test3 = train_test_split(X, y, test_size=0.2, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

print('Train size:     ', len(X_tr))
print('Validation size:', len(X_val))
print('Test size:      ', len(X_test3))

# tuning max_depth on validation set
best_depth = 1
best_val = 0.0

for depth in [1, 2, 3, 4, 5]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_tr, y_tr)
    val_acc = m.score(X_val, y_val)
    print('max_depth =', depth, '-> Validation Accuracy:', round(val_acc, 2))
    if val_acc > best_val:
        best_val = val_acc
        best_depth = depth

print('\nBest max_depth selected:', best_depth)
final_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42).fit(X_tr, y_tr)
print('Final Model Test Accuracy:', round(final_model.score(X_test3, y_test3), 2))

### Task 4: Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

dt_cv = DecisionTreeClassifier(max_depth=3, random_state=42)
cv_scores = cross_val_score(dt_cv, X, y, cv=5)

print('5-Fold CV Scores:', [round(s, 2) for s in cv_scores])
print('Mean CV Accuracy:', round(cv_scores.mean(), 2))

### Task 5: Bagging vs Boosting (Ensemble Learning)

- **Bagging:** Builds models in parallel on random data subsets with replacement to reduce variance.
- **Boosting:** Builds models sequentially where each new model fixes mistakes from previous models to reduce bias.

In [ ]:
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier

bag_model = BaggingClassifier(n_estimators=50, random_state=42).fit(X_train, y_train)
boost_model = AdaBoostClassifier(n_estimators=50, random_state=42).fit(X_train, y_train)

print('Bagging Classifier Accuracy: ', round(bag_model.score(X_test, y_test), 2))
print('AdaBoost Classifier Accuracy:', round(boost_model.score(X_test, y_test), 2))

### Task 6: Random Forest & Feature Importances

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_train, y_train)

print('Model Accuracy Comparison:')
print('Single Decision Tree:', round(dt.score(X_test, y_test), 2))
print('Bagging Classifier:  ', round(bag_model.score(X_test, y_test), 2))
print('Random Forest:       ', round(rf_model.score(X_test, y_test), 2))

print('\nRandom Forest Feature Importances:')
for feature, imp in zip(X.columns, rf_model.feature_importances_):
    print(feature, ':', round(imp, 4))